# Experimental test 5 Result

* vllm기준으로 accuracy와 f1 score를 테스트 (vllm + llama)
* 옵션 수정으로 인해서 결과가 다 동일하게 나오는지 확인하기 위해서 테스트 진행 
* Self_Consistency_re로 테스트
    * fewshot 개수 : 3*3 = 9
    * fewshot 테스트에 활용할 질문의 개수  : 30
    * 소스코드 포함여부  : 'Y'           
    * 반복횟수 : 100회                
    * 시스템프롬프트 'sys_prompt10'
    * self-consistency 횟수 : 5
    * temperature : 0.01
    * 엑셀버전 : 'ver7'
* 이후 결과에 대해서 스코어 비교 진행 
* stop        = ["</Difficulty Level>"] 있는 버전의 점수


In [1]:
import os
import pandas as pd
from config import config as conf
import re
import numpy as np
from sklearn import metrics



In [2]:
def sc_calc_acc_condition_with_temp_with_sc(llm_model, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')]

    df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            print(f'size of the dataset : {df_eval.shape[0]}')
            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list


In [3]:
    # task('vq',              # llm_model
    #     3,                # few_shot_n
    #     59,                # test_n(# of question for test)
    #     'Y',              # q_src_yn 
    #     5,                # iteration num
    #     'sys_prompt10',   # prompt ver
    #     5,                # self-consistency number
    #     0.01,             # temperature
    #     'ver7'            # excel_verion
    #     )

In [4]:
# sc_vl_result_3_60_Y_100_sys_prompt10_5_0.01_ver7_99.csv
# stop        = ["</Difficulty Level>"] 있는 버전의 점수 
list_ =         sc_calc_acc_condition_with_temp_with_sc('vq', 3, 59, 'Y', 5, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

size of the dataset : 59
size of the dataset : 59
size of the dataset : 59
size of the dataset : 59
size of the dataset : 59
              precision    recall  f1-score   support

           0      0.989     0.652     0.786       138
           1      0.665     0.865     0.752       126
           2      0.600     0.774     0.676        31

    accuracy                          0.756       295
   macro avg      0.751     0.764     0.738       295
weighted avg      0.810     0.756     0.760       295

vq_result_3_59_Y :  75.59322033898304
[np.float64(74.57627118644068), np.float64(72.88135593220339), np.float64(76.27118644067797), np.float64(77.96610169491525), np.float64(76.27118644067797)]


In [ ]:
# # stop        = ["</Difficulty Level>"] 없는 버전 테스트

#     task('vq',              # llm_model
#         3,                # few_shot_n
#         58,                # test_n(# of question for test)
#         'Y',              # q_src_yn 
#         5,                # iteration num
#         'sys_prompt10',   # prompt ver
#         5,                # self-consistency number
#         0.01,             # temperature
#         'ver7'            # excel_verion

#         )

In [5]:
list_ =         sc_calc_acc_condition_with_temp_with_sc('vq', 3, 58, 'Y', 5, 'sys_prompt10', 5,  0.01, 'ver7')
print(list_)

size of the dataset : 57
size of the dataset : 57
size of the dataset : 56
size of the dataset : 58
size of the dataset : 56
              precision    recall  f1-score   support

           0      0.989     0.679     0.805       137
           1      0.680     0.860     0.759       121
           2      0.568     0.808     0.667        26

    accuracy                          0.768       284
   macro avg      0.746     0.782     0.744       284
weighted avg      0.819     0.768     0.773       284

vq_result_3_58_Y :  76.76056338028168
[np.float64(78.94736842105263), np.float64(80.7017543859649), np.float64(80.35714285714286), np.float64(68.96551724137932), np.float64(75.0)]
